# Figures for: Benchmarking Tools for Identification of rRNA Modifications

| Figure | Description |
|--------|------------|
| **Fig. 1** | Read length distributions (native vs IVT, 16S and 23S) |
| **Fig. 2** | Coverage bias toward 3ʹ end (linear + log, with 7 bp smoothing) |
| **Supp. Fig. S1** | Subsampled coverage comparisons |
| **Supp. Fig. S2** | Read quality vs length scatter plots |

In [ ]:
# Import libraries

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
from pathlib import Path

## Configuration

In [ ]:
# File paths
BASE_DIR = Path(".")
DEPTH_DIR = BASE_DIR / "depths"
STATS_DIR = BASE_DIR / "nanoplot_rawdata"
OUTPUT_DIR = BASE_DIR / "plots"
DEPTH_FILTERED_DIR = BASE_DIR / "subsampled_coverage_depths"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Reference lengths
LEN_16S = 1542
LEN_23S = 2904

# start and end for extended references (with 200 nt padding on each side)
RNA_START = 201
RNA_END_16S = 1742
RNA_END_23S = 3104

# Replicate colours (consistent across all figures)
REP_COLORS = ['#0072B2', '#D55E00', '#009E73']  # blue, red, orange
REP_LABELS = ['Rep 1', 'Rep 2', 'Rep 3']

# Font sizes
PANEL_LABEL_SIZE = 25
AXIS_LABEL_SIZE = 18
TICK_LABEL_SIZE = 18
TITLE_SIZE = 12

## Load data

In [ ]:
# Depth files loading function (samtools depth output: chrom, position, depth)
def load_depth(kind, rna, rep):
    """Load a depth file. kind='native'|'ivt', rna='16s'|'23s', rep=1|2|3."""
    fname = f"k12_{kind}_bc{rep}_mapped_{rna}_sorted.txt"
    df = pd.read_csv(f"{DEPTH_DIR}/{fname}", sep='\t', header=None)
    df.columns = ['chrom', 'position', 'depth']
    return df

depth = {}
for kind in ['native', 'ivt']:
    for rna in ['16s', '23s']:
        for rep in [1, 2, 3]:
            depth[(kind, rna, rep)] = load_depth(kind, rna, rep)

print(f"Loaded {len(depth)} depth files")
print(f"Example shape: {depth[('native', '16s', 1)].shape}")

In [ ]:
def smooth_depth(df, window=7):
    """Apply rolling mean smoothing to depth values."""
    df = df.copy()
    df['depth_smooth'] = df['depth'].rolling(window=window, center=True, min_periods=1).mean()
    return df

def load_depth_at_coverage(kind, rna, rep, cov):
    """Load depth file for a specific coverage level."""
    base = DEPTH_FILTERED_DIR / f"{cov}x_coverage_depth"
    fname = f"{kind}_{rna}_rep{rep}.txt"
    return pd.read_csv(base / kind / rna / fname, sep='\t', header=None,
                        names=['chrom', 'position', 'depth'])

def get_coverage_summary_filtered(kind, rna, cov, window=7):
    """Mean across replicates for a given coverage level, with smoothing."""
    dfs = []
    for rep in [1, 2, 3]:
        df = smooth_depth(load_depth_at_coverage(kind, rna, rep, cov), window=window)
        dfs.append(df.set_index('position')['depth_smooth'])
    combined = pd.concat(dfs, axis=1)
    return pd.DataFrame({
        'position': combined.index,
        'mean': combined.mean(axis=1),
    })

In [ ]:
# NanoPlot stats file loading function (pickle files with 'lengths' and 'quals' columns)
def load_stats(kind, rna, rep):
    """Load a NanoPlot pickle. kind='native'|'ivt', rna='16s'|'23s', rep=1|2|3."""
    fname = f"k12_{kind}_bc{rep}_mapped_{rna}-NanoPlot-data.pickle"
    return pd.read_pickle(f"{STATS_DIR}/{fname}")

stats = {}
for kind in ['native', 'ivt']:
    for rna in ['16s', '23s']:
        for rep in [1, 2, 3]:
            stats[(kind, rna, rep)] = load_stats(kind, rna, rep)

print(f"Loaded {len(stats)} stats files")
print(f"Example shape: {stats[('native', '16s', 1)].shape}")
print(f"Columns: {stats[('native', '16s', 1)].columns.tolist()}")

---
## Figure 1 — Read length distributions

Native vs IVT for 16S (panel A) and 23S (panel B). Each panel: native on top row, IVT on bottom row, 3 replicates as columns. Dashed red line marks full-length transcript.

**Subsampled** to 9,000 reads per replicate. Bin width = 50 bp.

In [ ]:
def plot_length_histograms():
    """Plot read length histograms for 16S (a) and 23S (b) as a single panel 4x3 figure.
    
    All reads plotted. Used stat='density' so replicates 
    with different read counts are visually comparable.
    """
    fig, axes = plt.subplots(4, 3, figsize=(16, 16), sharex=True)
    sns.set_theme(style='ticks')
    binwidth = 50

    # Row mapping: 0=native 16S, 1=IVT 16S, 2=native 23S, 3=IVT 23S
    row_config = [
        ('native', '16s', LEN_16S),
        ('ivt',    '16s', LEN_16S),
        ('native', '23s', LEN_23S),
        ('ivt',    '23s', LEN_23S),
    ]

    for row, (kind, rna, ref_len) in enumerate(row_config):
        for col, rep in enumerate([1, 2, 3]):
            ax = axes[row, col]
            data = stats[(kind, rna, rep)]['lengths']  # all reads, no subsampling
            bins = np.arange(data.min(), data.max() + binwidth, binwidth)

            sns.histplot(x=data, color=REP_COLORS[col], ax=ax, bins=bins,
                         stat='density', edgecolor='none', alpha=0.85)
            ax.axvline(x=ref_len, color='red', linestyle='--', linewidth=1)
            ax.tick_params(labelsize=TICK_LABEL_SIZE)
            ax.tick_params(axis='x', which='both', bottom=True, labelbottom=(row in [0, 1, 2, 3]))

            # Row labels (left column only)
            if col == 0:
                ax.set_ylabel('Density', fontsize=AXIS_LABEL_SIZE, labelpad=10)
            else:
                ax.set_ylabel('')

            # Bottom of each 2-row panel gets x-label
            if row in [1, 3]:
                ax.set_xlabel('Length (base pairs)', fontsize=AXIS_LABEL_SIZE, labelpad=10)
            else:
                ax.set_xlabel('')

    axes[0, 0].set_xlim(0, 3000)
    
    # Share y-axis within each panel group only
    # 16S: rows 0-1
    ymax_16s = max(axes[r, c].get_ylim()[1] for r in [0, 1] for c in range(3))
    for r in [0, 1]:
        for c in range(3):
            axes[r, c].set_ylim(0, ymax_16s)

    # 23S: rows 2-3
    ymax_23s = max(axes[r, c].get_ylim()[1] for r in [2, 3] for c in range(3))
    for r in [2, 3]:
        for c in range(3):
            axes[r, c].set_ylim(0, ymax_23s)

    # Hide y-tick labels on middle and right columns (keep ticks)
    for row in range(4):
        for col in [1, 2]:
            axes[row, col].tick_params(axis='y', labelleft=False, left=True)

    # Panel labels — positioned outside the plot area with offset
    # Using axes coordinates so label sits above-left of the top-left panel of each group
    axes[0, 0].text(-0.34, 1.2, 'a', fontsize=PANEL_LABEL_SIZE,
                     fontweight='bold', va='top', ha='left',
                     transform=axes[0, 0].transAxes)
    axes[2, 0].text(-0.34, 1.2, 'b', fontsize=PANEL_LABEL_SIZE,
                     fontweight='bold', va='top', ha='left',
                     transform=axes[2, 0].transAxes)

    # Add space between the two panels (16S and 23S)
    plt.subplots_adjust(hspace=0.4)

    # Tighten within each group but keep the gap between a and b
    fig.tight_layout()
    # Re-apply the inter-panel gap after tight_layout
    plt.subplots_adjust(hspace=0.35)

    return fig


fig1 = plot_length_histograms()
fig1.savefig(f"{OUTPUT_DIR}/Fig1_read_lengths.png", bbox_inches='tight', dpi=600)
fig1.savefig(f"{OUTPUT_DIR}/Fig1_read_lengths.pdf", bbox_inches='tight')
plt.show()

print("Figure 1 saved.")

---
## Figure 2 — Coverage bias toward the 3ʹ end

Native (top) and IVT (bottom) for 16S (panel A) and 23S (panel B). Each panel has linear y-axis (left) and log y-axis (right). 7 bp rolling-mean smoothing applied. Three replicates overlaid.

**No subsampling** — plots all positions from depth files.

In [ ]:
CONDITION_COLORS = {
    'native': '#0072B2',  # blue
    'ivt':    '#D55E00',  # vermillion
}

def get_coverage_summary(kind, rna, window=7):
    """Compute mean and min/max depth across replicates."""
    dfs = []
    for rep in [1, 2, 3]:
        df = smooth_depth(depth[(kind, rna, rep)], window=window)
        dfs.append(df.set_index('position')['depth_smooth'])
    
    combined = pd.concat(dfs, axis=1)
    return pd.DataFrame({
        'position': combined.index,
        'mean': combined.mean(axis=1),
        'min': combined.min(axis=1),
        'max': combined.max(axis=1),
    })


def plot_coverage():
    """Plot coverage bias for 16S (a) and 23S (b) as a single 4x2 figure.
    Shows mean coverage across replicates with min-max shaded band."""
    fig, axes = plt.subplots(4, 2, figsize=(16, 16), sharex=False)
    sns.set_theme(style='ticks')

    row_config = [
        ('native', '16s'),
        ('ivt',    '16s'),
        ('native', '23s'),
        ('ivt',    '23s'),
    ]

    for row, (kind, rna) in enumerate(row_config):
        summary = get_coverage_summary(kind, rna)
        row_label = 'Native' if kind == 'native' else 'IVT'

        for col, (yscale, ylabel_suffix) in enumerate([('linear', ''), ('log', ' (log)')]):
            ax = axes[row, col]

            y_mean = summary['mean']
            y_min = summary['min']
            y_max = summary['max']

            if yscale == 'log':
                y_mean = y_mean + 1
                y_min = y_min + 1
                y_max = y_max + 1

            colour = CONDITION_COLORS[kind]

            ax.plot(summary['position'], y_mean,
                    color=colour, linewidth=1.2, label=row_label)
            ax.fill_between(summary['position'], y_min, y_max,
                            color=colour, alpha=0.25)

            if yscale == 'log':
                ax.set_yscale('log')

            ref_start = 201
            ref_end = 1742 if rna == '16s' else 3104
            ax.axvline(x=ref_start, color='black', linestyle='--', linewidth=0.8)
            ax.axvline(x=ref_end, color='black', linestyle='--', linewidth=0.8)
            ax.tick_params(labelsize=TICK_LABEL_SIZE)
            ax.tick_params(axis='x', which='both', bottom=True,
                           labelbottom=(row in [1, 3]))

        axes[row, 0].set_ylabel('Coverage', fontsize=AXIS_LABEL_SIZE, labelpad=10)
        axes[row, 1].set_ylabel('Log coverage', fontsize=AXIS_LABEL_SIZE, labelpad=10)

    # X-labels
    for col in [0, 1]:
        axes[1, col].set_xlabel('Position on 16S rRNA (bp)', fontsize=AXIS_LABEL_SIZE, labelpad=10)
        axes[3, col].set_xlabel('Position on 23S rRNA (bp)', fontsize=AXIS_LABEL_SIZE, labelpad=10)

    # Panel labels
    axes[0, 0].text(-0.34, 1.2, 'a', fontsize=PANEL_LABEL_SIZE,
                     fontweight='bold', va='top', ha='left',
                     transform=axes[0, 0].transAxes)
    axes[2, 0].text(-0.34, 1.2, 'b', fontsize=PANEL_LABEL_SIZE,
                     fontweight='bold', va='top', ha='left',
                     transform=axes[2, 0].transAxes)

    fig.tight_layout()
    plt.subplots_adjust(hspace=0.35)
    return fig


fig2 = plot_coverage()
fig2.savefig(f"{OUTPUT_DIR}/Fig2_coverage.png", bbox_inches='tight', dpi=600)
fig2.savefig(f"{OUTPUT_DIR}/Fig2_coverage.pdf", bbox_inches='tight')
plt.show()

print("Figure 2 saved.")

---
## Supplementary Figure S2 — Read quality vs. read length

Native (top) and IVT (bottom) for 16S (panel A) and 23S (panel B). Scatter plot with alpha=0.05 to show density.

In [ ]:
def plot_quality_scatter():
    """Plot quality-vs-length for 16S (a) and 23S (b) as a single 4x3 figure.
    
    No subsampling — all reads plotted with adaptive alpha based on read count.
    """
    fig, axes = plt.subplots(4, 3, figsize=(16, 16), sharex=True, sharey=True)
    sns.set_theme(style='ticks')

    row_config = [
        ('native', '16s'),
        ('ivt',    '16s'),
        ('native', '23s'),
        ('ivt',    '23s'),
    ]

    for row, (kind, rna) in enumerate(row_config):
        for col, rep in enumerate([1, 2, 3]):
            ax = axes[row, col]
            data = stats[(kind, rna, rep)]

            n_reads = len(data)
            alpha = max(0.01, min(0.15, 5000 / n_reads))

            ax.scatter(data['lengths'], data['quals'],
                       c=REP_COLORS[col], s=2, alpha=alpha,
                       edgecolors='none', rasterized=True)

            ax.tick_params(labelsize=TICK_LABEL_SIZE)
            ax.tick_params(axis='x', which='both', bottom=True,
                           labelbottom=(row in [1, 3]))

            if col == 0:
                ax.set_ylabel('Quality score (Phred)', fontsize=AXIS_LABEL_SIZE,
                              labelpad=10)
            else:
                ax.set_ylabel('')

            if row in [1, 3]:
                ax.set_xlabel('Length (base pairs)', fontsize=AXIS_LABEL_SIZE,
                              labelpad=10)
            else:
                ax.set_xlabel('')

    axes[0, 0].set_xlim(0, 3000)
    axes[0, 0].set_ylim(0, 16)
    axes[0, 0].set_xticks([0, 1000, 2000, 3000])
    axes[0, 0].set_yticks([0, 4, 8, 12, 16])

    # Panel labels
    axes[0, 0].text(-0.34, 1.2, 'a', fontsize=PANEL_LABEL_SIZE,
                     fontweight='bold', va='top', ha='left',
                     transform=axes[0, 0].transAxes)
    axes[2, 0].text(-0.34, 1.2, 'b', fontsize=PANEL_LABEL_SIZE,
                     fontweight='bold', va='top', ha='left',
                     transform=axes[2, 0].transAxes)

    fig.tight_layout()
    plt.subplots_adjust(hspace=0.35)
    return fig


figs2 = plot_quality_scatter()
figs2.savefig(f"{OUTPUT_DIR}/SuppFigS2_quality_vs_length.png", bbox_inches='tight', dpi=600)
figs2.savefig(f"{OUTPUT_DIR}/SuppFigS2_quality_vs_length.pdf", bbox_inches='tight')
plt.show()

print("Supplementary Figure S2 saved.")

## Supplementary Figure S1 — Subsampled coverage comparison


In [ ]:
def plot_filtered_coverages():
    """Supp Fig S1: Filtered coverage at multiple depths."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    sns.set_theme(style='ticks')

    panel_config = [
        (0, 0, 'native', '16s', 'a'),
        (0, 1, 'ivt',    '16s', ''),
        (1, 0, 'native', '23s', 'b'),
        (1, 1, 'ivt',    '23s', ''),
    ]

    coverage_levels = [10, 25, 50, 100, 200, 500, 1000]
    cmap = plt.cm.plasma  # more contrast than viridis
    colors = {cov: cmap(i / (len(coverage_levels) - 1)) for i, cov in enumerate(coverage_levels)}

    for row, col, kind, rna, label in panel_config:
        ax = axes[row, col]

        for i, cov in enumerate(coverage_levels):
            summary = get_coverage_summary_filtered(kind, rna, cov)
            ax.plot(summary['position'], summary['mean'], color=colors[cov], linewidth=1, label=f'{cov}×', alpha=0.9)

        ref_end = RNA_END_16S if rna == '16s' else RNA_END_23S
        ax.axvline(x=RNA_START, color='black', linestyle='--', linewidth=0.8)
        ax.axvline(x=ref_end, color='black', linestyle='--', linewidth=0.8)

        rna_label = '16S' if rna == '16s' else '23S'
        kind_label = 'Native' if kind == 'native' else 'IVT'
        #ax.set_title(f'{kind_label} — {rna_label}', fontsize=TITLE_SIZE)
        ax.set_ylabel('Coverage', fontsize=AXIS_LABEL_SIZE, labelpad=10)
        ax.set_xlabel(f'Position on {rna_label} rRNA (bp)', fontsize=AXIS_LABEL_SIZE, labelpad=10)
        ax.set_yscale('log')
        ax.tick_params(labelsize=TICK_LABEL_SIZE)

        ax.text(-0.32, 1.2, label, fontsize=PANEL_LABEL_SIZE,
                fontweight='bold', va='top', ha='left',
                transform=ax.transAxes)

    # Remove individual legends from all panels (each panel adds duplicate labels)
    for row in range(2):
        for col in range(2):
            legend = axes[row, col].get_legend()
            if legend:
                legend.remove()

    # Single horizontal legend below the figure
    handles, labels = axes[0, 0].get_legend_handles_labels()
    legend_handles = [copy.copy(h) for h in handles]
    # Make legend lines thicker
    for h in legend_handles:
        h.set_linewidth(4)
    
    fig.legend(legend_handles, labels, title='',
               loc='lower center', ncol=len(coverage_levels),
               fontsize=TICK_LABEL_SIZE, frameon=False,
               bbox_to_anchor=(0.5, -0.06))
    
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.1)
    
    return fig

figs1 = plot_filtered_coverages()
figs1.savefig(f"{OUTPUT_DIR}/SuppFigS1_subsampled_coverage_comparison.png", bbox_inches='tight', dpi=600)
figs1.savefig(f"{OUTPUT_DIR}/SuppFigS1_subsampled_coverage_comparison.pdf", bbox_inches='tight')
plt.show()

print("Supplementary Figure S1 saved.")

---
## Summary of outputs

All figures saved to `OUTPUT_DIR` in both PNG (600 dpi) and PDF formats.

In [ ]:
print("Generated files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024
    print(f"  {f:50s} {size_kb:8.1f} KB")